In [2]:
import json
import re
from pathlib import Path


def replace_footnotes(entry):
    contents = entry.get("contents", "")
    footnotes = entry.get("footnotes", {})

    def replacer(match):
        key = match.group(1)  # Extract the number inside {}
        return_val = footnotes.get(key, match.group(0)) # Replace or keep original
        return " " + return_val + " "  # Add spaces around the footnote  
    return re.sub(r'\{(\d+)\}', replacer, contents)

def normalize_text(text: str) -> str:
    """
    Normalize whitespace while preserving paragraph structure.
    """

    # Convert all line endings to Unix style
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Replace tabs with spaces
    text = text.replace("\t", " ")

    # Remove trailing spaces at end of lines
    text = re.sub(r"[ \t]+$", "", text, flags=re.MULTILINE)

    # Collapse multiple spaces inside a line
    text = re.sub(r"[ ]{2,}", " ", text)

    # Remove spaces before newlines
    text = re.sub(r" +\n", "\n", text)

    # Collapse 3+ blank lines into exactly 2
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove leading/trailing whitespace
    text = text.strip()

    return text

In [ ]:
# Open and read a JSON file
with open('./Data/CWMG.json', 'r') as file:
    data = json.load(file)

for entry in data:
        entry["contents"] = replace_footnotes(entry)

json.dump(data, open('./Mahatma Gandhi Data Corpus/CWMG_processed.json', 'w'), indent=4)

In [7]:
with open('./Data/CWMG_processed.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

parts = []
for entry in data:
    contents = replace_footnotes(entry)  # includes footnote replacement
    document_date = entry.get("document_date", "")
    title = entry.get("title", "")
    parts.append(
        f"The document titled '{title}' was created on {document_date}. "
        f"The content of the document with the title '{title}' is as follows: {contents}. "
        f"The document with the content '{contents}' was created on {document_date}. \n"
    )

parts = [normalize_text(part) for part in parts]

with open('./Data/CWMG.txt', 'w', encoding='utf-8') as file:
    file.write("".join(parts))

# Cleaning the autobiography text file

In [14]:
import re
import ftfy

def clean_text(input_file_path, output_file_path):
    with open(input_file_path, 'r', encoding='utf-8') as f:
        text = f.read()

    # 1. Fix encoding/mojibake issues
    text = ftfy.fix_text(text)

    # 2. Remove Form Feeds
    text = re.sub(r'\x0c', '', text)

    # 3. Remove Roman Numeral Chapter Headers
    roman_header_pattern = (
        r'^\s*(?=[MDCLXVI])M{0,4}(?:CM|CD|D?C{0,3})(?:XC|XL|L?X{0,3})'
        r'(?:IX|IV|V?I{0,3})\s*$\n+\s*[A-Z0-9\s\,\'\-\?]+$'
    )
    text = re.sub(roman_header_pattern, '', text, flags=re.MULTILINE)

    # 4. Remove Generic Headers/Footers with Page Numbers
    TITLE = r'[A-Z][A-Z\s\,\'\-\?]{2,}'
    p1 = rf'^\s*\d+\s+{TITLE}\s*$'
    p2 = rf'^\s*\d+\s*\n+\s*{TITLE}\s*$'
    p3 = rf'^\s*{TITLE}\s+\d+\s*$'
    p4 = rf'^\s*{TITLE}\s*\n+\s*\d+\s*$'
    generic_header_pattern = f"(?:{p1}|{p2}|{p3}|{p4})"
    text = re.sub(generic_header_pattern, '', text, flags=re.MULTILINE)

    # 5. Remove Standalone Page Numbers
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # 6. Fix Hyphenated Words Across Line Breaks
    text = re.sub(r'(\w+)-\n\s*(\w+)', r'\1\2', text)

    # ---------------------------------------------------------
    # NEW STEP: Merge lines starting with lowercase letters
    # ---------------------------------------------------------
    # Matches a newline followed by any lowercase character [a-z]
    text = re.sub(r'\n(?=[a-z])', ' ', text)

    # 7. Collapse multiple spaces into 1
    text = re.sub(r'[ \t]+', ' ', text)

    # 8. Normalize empty lines (max 2 newlines for paragraph breaks)
    text = re.sub(r'\n{3,}', '\n\n', text)


    cleaned_text = ftfy.fix_text(text)

    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write(cleaned_text.strip())

    print(f"Cleaning complete! Saved to {output_file_path}")

clean_text('./Data/autobiography.txt', './Data/autobiography_cleaned.txt')

Cleaning complete! Saved to ./Data/autobiography_cleaned.txt


In [1]:

def merge_files(file1_path, file2_path, output_path):
    with open(file1_path, 'r', encoding='utf-8') as f1, \
         open(file2_path, 'r', encoding='utf-8') as f2, \
         open(output_path, 'w', encoding='utf-8') as out:
        
        # Write first file
        content1 = f1.read()
        out.write(content1)
        
        # Ensure there is a newline between files if file1 doesn't end with one
        if not content1.endswith('\n'):
            out.write('\n')
            
        # Write second file
        out.write(f2.read())

    print(f"Files successfully combined into {output_path}")

# Example usage:
merge_files('./Data/autobiography_cleaned.txt', './Data/CWMG.txt', './Data/combined_data.txt')


Files successfully combined into ./Data/combined_data.txt
